In [1]:
%pip install "scikit-learn>=1.2" pandas numpy scipy 

   ━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━ 5.8/11.1 MB 80.8 kB/s eta 0:01:07m
Resuming download scikit_learn-1.6.1-cp39-cp39-macosx_12_0_arm64.whl (5.8 MB/11.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 6.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 12.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer,
)
from sklearn.impute import SimpleImputer

print("Библиотеки загружены")

Библиотеки загружены


In [5]:
from pathlib import Path
import pandas as pd

file_path = (
    Path.home()
    / "Desktop"
    / "ML"
    / "ML players-eda-pw-3"
    / "data"
    / "players.csv"
)

print("Файл найден:", file_path.exists())

df = pd.read_csv(file_path)

print("Размер датасета:", df.shape)
display(df.head())

Файл найден: True
Размер датасета: (19789, 60)


,player_id,common_name,first_name,last_name,overall_rating,position,alternate_positions,club,league,nationality,...,defending_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,edition,snapshot_date
0,227203,Alexia Putellas,Alexia,Putellas Segura,91,CM,CAM ST,London City,Barclays WSL,Spain,...,60,82,64,15,17,11,15,10,fc27,2026-09-12
1,231747,NaN,Kylian,Mbappé,91,ST,LW,Real Madrid,LALIGA EA SPORTS,France,...,15,34,24,13,5,7,11,6,fc27,2026-09-12
2,239085,NaN,Erling,Haaland,91,ST,NaN,Manchester City,Premier League,Norway,...,42,47,33,7,14,13,11,7,fc27,2026-09-12
3,192119,NaN,Thibaut,Courtois,90,GK,NaN,Real Madrid,LALIGA EA SPORTS,Belgium,...,20,18,16,87,89,78,90,90,fc27,2026-09-12
4,202126,NaN,Harry,Kane,90,ST,NaN,FC Bayern München,Bundesliga,England,...,46,44,42,8,10,11,14,11,fc27,2026-09-12


In [6]:
train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
)

print("Train:", train.shape)
print("Test:", test.shape)

assert set(train.index).isdisjoint(set(test.index))
print("Индексы train и test не пересекаются")

Train: (15831, 60)
Test: (3958, 60)
Индексы train и test не пересекаются


In [7]:
feature_info = pd.DataFrame({
    "тип": train.dtypes.astype(str),
    "пропусков": train.isna().sum(),
    "доля_пропусков_%": (
        train.isna().mean() * 100
    ).round(2),
})

display(feature_info)

print("Столбцы с пропусками:")
display(feature_info.loc[feature_info["пропусков"] > 0])

,тип,пропусков,доля_пропусков_%
player_id,int64,0,0.00
common_name,object,13544,85.55
first_name,object,0,0.00
last_name,object,0,0.00
overall_rating,int64,0,0.00
position,object,0,0.00
alternate_positions,object,5502,34.75
club,object,0,0.00
league,object,0,0.00
nationality,object,0,0.00


Столбцы с пропусками:


,тип,пропусков,доля_пропусков_%
common_name,object,13544,85.55
alternate_positions,object,5502,34.75
height_cm,float64,15831,100.00
weight_kg,float64,15831,100.00
playstyles,object,9411,59.45


In [8]:
# Числовые поля, которые не будем использовать.
excluded_numeric = [
    "player_id",
    "height_cm",
    "weight_kg",
]

numeric_cols = [
    column
    for column in train.select_dtypes(include="number").columns
    if column not in excluded_numeric
]

# Этот признак будет создан внутри Pipeline.
numeric_cols = numeric_cols + ["age_years"]

categorical_cols = [
    "position",
    "alternate_positions",
    "club",
    "league",
    "nationality",
    "gender",
    "preferred_foot",
    "playstyles",
]

print("Числовых признаков:", len(numeric_cols))
print("Категориальных признаков:", len(categorical_cols))

print("\nЧисловые признаки:")
print(numeric_cols)

print("\nКатегориальные признаки:")
print(categorical_cols)

Числовых признаков: 44
Категориальных признаков: 8

Числовые признаки:
['overall_rating', 'skill_moves', 'weak_foot', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physicality', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control', 'movement_acceleration', 'movement_sprint_speed', 'movement_agility', 'movement_reactions', 'movement_balance', 'power_shot_power', 'power_jumping', 'power_stamina', 'power_strength', 'power_long_shots', 'mentality_aggression', 'mentality_interceptions', 'mentality_positioning', 'mentality_vision', 'mentality_penalties', 'mentality_composure', 'defending_awareness', 'defending_standing_tackle', 'defending_sliding_tackle', 'goalkeeping_diving', 'goalkeeping_handling', 'goalkeeping_kicking', 'goalkeeping_positioning', 'goalkeeping_reflexes', 'age_years']

Категориальные признаки:
['p

In [9]:
strategy_table = pd.DataFrame(
    [
        {
            "признак": column,
            "группа": "числовой / порядковый",
            "пропуски": "медиана train",
            "преобразование": "StandardScaler",
        }
        for column in numeric_cols
    ]
    +
    [
        {
            "признак": column,
            "группа": "категориальный",
            "пропуски": "отдельная категория: неизвестно",
            "преобразование": "OneHotEncoder",
        }
        for column in categorical_cols
    ]
)

display(strategy_table)

,признак,группа,пропуски,преобразование
0,overall_rating,числовой / порядковый,медиана train,StandardScaler
1,skill_moves,числовой / порядковый,медиана train,StandardScaler
2,weak_foot,числовой / порядковый,медиана train,StandardScaler
3,pace,числовой / порядковый,медиана train,StandardScaler
4,shooting,числовой / порядковый,медиана train,StandardScaler
5,passing,числовой / порядковый,медиана train,StandardScaler
6,dribbling,числовой / порядковый,медиана train,StandardScaler
7,defending,числовой / порядковый,медиана train,StandardScaler
8,physicality,числовой / порядковый,медиана train,StandardScaler
9,attacking_crossing,числовой / порядковый,медиана train,StandardScaler


In [10]:
# Преобразование дат не использует статистики других строк.
def add_age(data):
    result = data.copy()

    birthdate = pd.to_datetime(
        result["birthdate"],
        errors="coerce",
    )

    snapshot_date = pd.to_datetime(
        result["snapshot_date"],
        errors="coerce",
    )

    result["age_years"] = (
        snapshot_date - birthdate
    ).dt.days / 365.25

    return result


# Обработка числовых признаков.
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
])


# Обработка категориальных признаков.
categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="неизвестно",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        ),
    ),
])


# Разные преобразования для разных групп столбцов.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_cols,
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_cols,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)


# Все шаги объединены в один объект.
pipeline = Pipeline(steps=[
    (
        "add_age",
        FunctionTransformer(add_age),
    ),
    (
        "preprocessor",
        preprocessor,
    ),
])

print("Pipeline создан")

Pipeline создан


In [11]:
# Единственное обучение пайплайна — только на train.
X_train_processed = pipeline.fit_transform(train)

# Получаем обученные внутренние объекты.
fitted_preprocessor = pipeline.named_steps["preprocessor"]

num_fitted = fitted_preprocessor.named_transformers_["num"]
cat_fitted = fitted_preprocessor.named_transformers_["cat"]

imputer = num_fitted.named_steps["imputer"]
scaler = num_fitted.named_steps["scaler"]
encoder = cat_fitted.named_steps["encoder"]

# Сохраняем параметры до обработки test.
medians_before = imputer.statistics_.copy()
means_before = scaler.mean_.copy()
variances_before = scaler.var_.copy()

categories_before = [
    categories.copy()
    for categories in encoder.categories_
]

# Применяем готовые преобразования к test.
# Здесь используется transform, а не fit_transform.
X_test_processed = pipeline.transform(test)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)

Train после обработки: (15831, 3140)
Test после обработки: (3958, 3140)


In [12]:
def all_values_finite(matrix):
    # У разреженной матрицы проверяем сохранённые значения.
    # Остальные элементы — нули.
    if sparse.issparse(matrix):
        return np.isfinite(matrix.data).all()

    return np.isfinite(np.asarray(matrix)).all()


# Проверка размерностей и количества строк.
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert X_train_processed.shape[0] == len(train)
assert X_test_processed.shape[0] == len(test)

# Проверка отсутствия NaN и бесконечностей.
assert all_values_finite(X_train_processed)
assert all_values_finite(X_test_processed)

# Проверка, что обработка test не изменила параметры.
assert np.array_equal(medians_before, imputer.statistics_)
assert np.array_equal(means_before, scaler.mean_)
assert np.array_equal(variances_before, scaler.var_)

assert all(
    np.array_equal(before, after)
    for before, after in zip(
        categories_before,
        encoder.categories_,
    )
)

# Дополнительно проверяем источник медиан:
# они должны совпадать с медианами train.
train_with_age = add_age(train)

expected_train_medians = (
    train_with_age[numeric_cols]
    .median()
    .to_numpy()
)

assert np.allclose(
    imputer.statistics_,
    expected_train_medians,
)

print("Количество признаков train и test совпадает")
print("Количество строк сохранено")
print("NaN и бесконечностей нет")
print("Медианы рассчитаны только по train")
print("Обработка test не изменила параметры пайплайна")
print("Все проверки пройдены")

Количество признаков train и test совпадает
Количество строк сохранено
NaN и бесконечностей нет
Медианы рассчитаны только по train
Обработка test не изменила параметры пайплайна
Все проверки пройдены


В работе использован датасет players.csv. До вычисления статистик данные разделены на обучающую и тестовую выборки в соотношении 80/20 с random_state=42. Обучающая выборка содержит 15 831 строку, тестовая — 3 958 строк.

Для всех используемых числовых характеристик выбрано заполнение пропусков медианой обучающей выборки. Характеристики различаются между игроками разных позиций, поэтому симметричность их распределений не предполагается. Медиана меньше зависит от крайних значений, чем среднее. Для порядковых оценок skill_moves и weak_foot сохранено числовое представление, поскольку значения отражают упорядоченные уровни. Для них также выбрана медиана. В текущем файле используемые исходные числовые характеристики не содержат пропусков, поэтому заполнение не изменяет их значения. После заполнения применяется StandardScaler, параметры которого рассчитываются только по train.

Признаки height_cm и weight_kg полностью пустые в обучающей выборке. Их среднее и медиану вычислить невозможно, поэтому они исключены. Произвольные значения роста и веса не подставляются.

Для каждого категориального признака — position, alternate_positions, club, league, nationality, gender, preferred_foot и playstyles — выбрано заполнение пропусков отдельной категорией «неизвестно». Это сохраняет факт отсутствия информации и не приписывает игроку наиболее частую категорию. Для alternate_positions отсутствие значения не интерпретируется автоматически как доказанное отсутствие дополнительных позиций. Аналогично пропуск playstyles не считается доказательством отсутствия игровых стилей.

Категориальные признаки преобразованы с помощью OneHotEncoder. Значения alternate_positions и playstyles рассматриваются как целые комбинации; отдельные элементы списков в этой реализации не выделяются. Параметр handle_unknown="ignore" позволяет применять обученный кодировщик к новым категориям test без изменения числа выходных столбцов.

Из birthdate и snapshot_date вычислен числовой признак age_years — возраст игрока на дату снимка данных. Он обрабатывается числовой веткой с заполнением медианой и масштабированием. Вычисление возраста использует только значения соответствующей строки. Исходные даты не включаются в выходную матрицу.

Идентификатор player_id и поля common_name, first_name, last_name исключены как идентификационные данные. Признак edition исключён, поскольку в данном файле представлен одной версией.

Все преобразования объединены в Pipeline с ColumnTransformer. Обучение выполнено методом fit_transform только на train. Для test использован метод transform. Итоговые размерности составили (15831, 3140) для train и (3958, 3140) для test. Проверки подтвердили отсутствие NaN и бесконечных значений, совпадение числа выходных признаков, соответствие медиан статистикам train и неизменность параметров после обработки test. Утечка статистик test в обучение предобработки не обнаружена.